# 07 - Night Shift Final Validation

## Purpose

Validate the complete Night Shift extension.

This notebook checks that all advanced Gold tables and the executive view exist, contain rows, and have clean reporting grains.

## Expected checks

- all Night Shift tables exist
- executive view returns rows
- row counts are greater than zero
- key dimensions are not null
- duplicate grain checks pass
- trust audit contains records
- observability summary contains records

## Main idea

Final validation proves that the Night Shift extension is implemented, explainable, and ready for presentation.

In [0]:
import sys
from pyspark.sql import Row
from pyspark.sql import functions as F

repo_src_path = "/Workspace/Users/dirella@startsteps.org/vattenfall-week9-capstone-anna/src"

if repo_src_path not in sys.path:
    sys.path.append(repo_src_path)

from validation.night_shift_checks import count_duplicate_grain, count_nulls

In [0]:
catalog = "vattenfall_dev"
schema = "analytics"

night_shift_tables = [
    f"{catalog}.{schema}.gold_data_trust_audit",
    f"{catalog}.{schema}.gold_asset_incident_intelligence",
    f"{catalog}.{schema}.gold_weather_grid_risk_correlation",
    f"{catalog}.{schema}.gold_market_operations_stress",
    f"{catalog}.{schema}.gold_pipeline_observability_summary",
    f"{catalog}.{schema}.gold_executive_energy_risk_dashboard_base",
]

night_shift_views = [
    f"{catalog}.{schema}.vw_executive_energy_risk_dashboard",
]

print("Night Shift tables:")
for table_name in night_shift_tables:
    print("-", table_name)

print("Night Shift views:")
for view_name in night_shift_views:
    print("-", view_name)

In [0]:
row_count_rows = []

for object_name in night_shift_tables + night_shift_views:
    df = spark.table(object_name)

    row_count_rows.append(
        Row(
            object_name=object_name,
            row_count=df.count(),
            column_count=len(df.columns),
            validation_status="PASS" if df.count() > 0 else "FAIL",
        )
    )

row_count_df = spark.createDataFrame(row_count_rows)

display(row_count_df)

In [0]:
empty_outputs = row_count_df.filter(F.col("row_count") <= 0)

if empty_outputs.count() > 0:
    display(empty_outputs)
    raise ValueError("Night Shift validation failed: one or more outputs are empty.")

print("Night Shift row count validation passed.")

In [0]:
null_checks = {
    f"{catalog}.{schema}.gold_asset_incident_intelligence": ["asset_id", "region"],
    f"{catalog}.{schema}.gold_weather_grid_risk_correlation": ["report_day", "region"],
    f"{catalog}.{schema}.gold_market_operations_stress": ["report_day", "region"],
    f"{catalog}.{schema}.gold_executive_energy_risk_dashboard_base": ["report_day", "region", "executive_risk_status"],
    f"{catalog}.{schema}.vw_executive_energy_risk_dashboard": ["report_day", "region", "executive_risk_status"],
}

null_rows = []

for object_name, columns in null_checks.items():
    df = spark.table(object_name)

    for column_name in columns:
        null_rows.append(
            Row(
                object_name=object_name,
                column_name=column_name,
                null_count=count_nulls(df, column_name),
            )
        )

null_df = spark.createDataFrame(null_rows)

display(null_df)

In [0]:
trust_audit_df = spark.table(f"{catalog}.{schema}.gold_data_trust_audit")
observability_df = spark.table(f"{catalog}.{schema}.gold_pipeline_observability_summary")

print("Trust audit rows:", trust_audit_df.count())
print("Observability rows:", observability_df.count())

print("Trust audit status distribution:")
trust_audit_df.groupBy("check_status").count().show()

print("Observability validation status distribution:")
observability_df.groupBy("validation_status").count().show()

if trust_audit_df.count() == 0:
    raise ValueError("Trust audit validation failed: no audit records found.")

if observability_df.count() == 0:
    raise ValueError("Observability validation failed: no observability records found.")

print("Trust audit and observability checks passed.")

In [0]:
executive_view = f"{catalog}.{schema}.vw_executive_energy_risk_dashboard"
executive_df = spark.table(executive_view)

print("Executive risk status distribution:")
executive_df.groupBy("executive_risk_status").count().show()

display(executive_df)

In [0]:
print("Night Shift final validation completed.")
print("All advanced Gold tables and the executive view are available, populated, and ready for presentation.")